# Module 4: Predictive Analyst Agent

This notebook verifies the Predictive Analyst Agent which performs pattern mining on historical cold stores, computes actual vs guidance KPI deltas, generates adversarial analyst questions, and gates them into answerable vs data gap lists using the Relevance Gate.

In [ ]:
# Cell 1: Initialization
import sys
from loguru import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

from extraction_agent import EmbeddingService
from predictive_analyst_agent import PredictiveAnalystAgent

embedding_service = EmbeddingService()
agent = PredictiveAnalystAgent(embedding_service)
print("Predictive Analyst Agent initialized successfully!")

In [ ]:
# Cell 2: Run Predictive Analyst Agent for Reliance Industries
print("Running Predictive Analyst Agent for Reliance Industries...")
results = await agent.run("Reliance Industries")
print("Execution complete!")

In [ ]:
# Cell 3: Display Top 10 Answerable Questions
print("=== TOP 10 PREDICTED ANSWERABLE QUESTIONS ===")
answerable = results["answerable"]
for idx, q in enumerate(answerable[:10]):
    print(f"\nQ{idx+1}: {q.question}")
    print(f"Topic: {q.topic} | Adversarial Score: {q.adversarial_score} | Sentiment: {q.sentiment}")
    print(f"Why Tough: {q.why_tough}")
    print(f"Source Quarters: {q.source_quarters}")
    print(f"Context Chunks Count: {len(q.hot_store_chunks)}")

In [ ]:
# Cell 4: Display Data Gaps
print("=== DETECTED DATA GAPS (UNANSWERABLE QUESTIONS) ===")
gaps = results["data_gaps"]
if not gaps:
    print("No data gaps detected (all questions are answerable with hot_store context).")
else:
    for idx, q in enumerate(gaps[:10]):
        print(f"\nGap Q{idx+1}: {q.question}")
        print(f"Topic: {q.topic} | Adversarial Score: {q.adversarial_score}")
        print(f"Gap Reason: {q.gap_reason}")

In [ ]:
# Cell 5: Display KPI Deltas
print("=== EXTRACTED KPI DELTA SUMMARY ===")
kpis = results["kpi_delta"].get("kpis", [])
if not kpis:
    print("No actual/guidance KPI pairs extracted.")
else:
    for k in kpis:
        status = "MISS" if k["is_miss"] else "BEAT"
        print(f"- Metric: {k['metric']} | Actual: {k['actual_value']}{k['unit']} | Guidance: {k['guided_value']}{k['unit']} | Delta: {k['delta_pct']:.2f}% | Status: {status}")